In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

analysis_df = pd.read_csv("../synthetic_esbl_data.csv")
#remove n_sites and n_samples columns if they exist
analysis_df = analysis_df.loc[:, ~analysis_df.columns.isin(['n_sites', 'n_samples'])]


In [17]:
prescriptions_df = pd.DataFrame({
    'subject': [1001, 1002],
    'admission_date': ['2024-01-01', '2024-02-15'],
    'discharge_date': ['2024-01-10', '2024-02-20'],
    'medication_name_short': ['Amoxicillin', 'Ciprofloxacin'],
    'therapeutical_class': ['Antibiotic', 'Antibiotic']
})

# Optional: convert dates to datetime
prescriptions_df['admission_date'] = pd.to_datetime(prescriptions_df['admission_date'])
prescriptions_df['discharge_date'] = pd.to_datetime(prescriptions_df['discharge_date'])

prescriptions_df.groupby(by = ['subject', 'admission_date'])['medication_name_short'].agg(list)

admnission_date    [Ciprofloxacin]
subject              [Amoxicillin]
Name: medication_name_short, dtype: object

In [11]:
comorbidities = ['anemia','asthma', 'cancer', 'copd', 'hypertension', 'ischaemic_heart_disease',
       'obesity', 'renal_failure', 'type2_diabetes']

asthma_drugs = [
    'salbutamol',
    'beclometasone',
    'budesonide',
    'fluticasone',
    'beclometasone-formoterol',
    'budesonide-formoterol',
    'fluticasone-salmeterol',
    'fluticasone-vilanterol',
    'fluticasone-formoterol'
]

copd_drugs = [
    'tiotropium',
    'glycopyrronium',
    'aclidinium',
    'umeclidinium',
    'glycopyrronium-indacaterol',
    'umeclidinium-vilanterol',
    'fluticasone/umeclidinium/vilanterol',
    'beclometasone/formoterol/glycopyrronium',
    'indacaterol',
    'salmeterol'
]

hypertension_drugs = [
    'amlodipine', 'nifedipine', 'felodipine',   # CCBs
    'lisinopril', 'perindopril',                # ACE inhibitors
    'losartan', 'irbesartan', 'candesartan',    # ARBs
    'bisoprolol', 'atenolol', 'metoprolol',     # beta-blockers
    'indapamide', 'bendroflumethiazide',        # thiazides
    'doxazosin'                                 # alpha-blocker
]

ihd_drugs = [
    'glyceryl trinitrate', 'isosorbide mononitrate',  # nitrates
    'aspirin', 'clopidogrel',                         # antiplatelets (if present)
    'atorvastatin', 'simvastatin',                    # statins (if present)
    'ranolazine'                                      # angina-specific
]

type2_diabetes_drugs = [
    'metformin',
    'gliclazide',
    'linagliptin',
    'sitagliptin',
    'alogliptin',
    'dapagliflozin',
    'empagliflozin',
    'canagliflozin',
    'semaglutide',
    'liraglutide',
    'dulaglutide'
]

ckd_drugs = [
    'sodium bicarbonate'   # metabolic acidosis in CKD
]
comorb_dict = {
    'asthma': asthma_drugs,
    'copd': copd_drugs,
    'hypertension': hypertension_drugs,
    'ischaemic_heart_disease': ihd_drugs,
    'renal_failure': ckd_drugs,
    'type2_diabetes': type2_diabetes_drugs
}

['asthma',
 'copd',
 'hypertension',
 'ischaemic_heart_disease',
 'renal_failure',
 'type2_diabetes']

In [23]:
def infer_comorb(prescriptions_df, medication_col, comorb_dict, subject_col='subject', admission_col='admission_date'):
    subject_medications = (
        prescriptions_df
        .groupby([subject_col, admission_col])[medication_col]
        .agg(list)
        .rename('medications')
        .reset_index()
    )

    # Normalize medication strings once to make matching case-insensitive.
    subject_medications['medications'] = subject_medications['medications'].apply(
        lambda meds: [str(m).strip().lower() for m in meds]
    )

    for comorb, meds in comorb_dict.items():
        meds_set = {str(m).strip().lower() for m in meds}
        subject_medications[comorb] = subject_medications['medications'].apply(
            lambda patient_meds: int(any(med in meds_set for med in patient_meds))
        )

    return subject_medications

In [24]:
infer_comorb(prescriptions_df, 'medication_name_short', comorb_dict)

,subject,admission_date,medications,asthma,copd,hypertension,ischaemic_heart_disease,renal_failure,type2_diabetes
0,1001,2024-01-01,[amoxicillin],0,0,0,0,0,0
1,1002,2024-02-15,[ciprofloxacin],0,0,0,0,0,0
